In [0]:
describe coffe_sales;

-- Find distinct coffee_name values
select distinct coffee_name from coffe_sales;

-- Find all transactions for a specific Weekday (e.g., 'Monday')
select * from coffe_sales WHERE Weekday = 'Mon';

-- count number of transactions per Month_name
select Month_name, count(*) as transaction_count
from coffe_sales
group by Month_name;

-- Calculate total sales (money) per coffee_name
select coffee_name, sum(money) as total_sales
from coffe_sales
group by coffee_name;

-- Find average spending per cash_type
select cash_type, avg(money) as avg_spending
from coffe_sales
group by cash_type;

-- Get top 5 highest transactions by money
select *
from coffe_sales
ORDER BY money DESC
LIMIT 5;

-- count transactions per hour_of_day
select hour_of_day, count(*) as transaction_count
from coffe_sales
group by hour_of_day;

-- Find busiest hour (highest number of transactions)
select hour_of_day, count(*) as transaction_count
from coffe_sales
group by hour_of_day
ORDER BY transaction_count DESC
LIMIT 1;

-- Calculate total sales per Weekday
select Weekday, sum(money) as total_sales
from coffe_sales
group by Weekday;

-- Count number of transactions per (coffee_name, cash_type)
select coffee_name, cash_type, count(*) as transaction_count
from coffe_sales
group by coffee_name, cash_type;

-- Find top-selling coffee by revenue
select coffee_name, sum(money) as total_revenue
from coffe_sales
group by coffee_name
order by total_revenue desc
limit 1;

-- Rank coffee types based on total revenue
select
  coffee_name,
  sum(money) as total_revenue,
  rank() over (order by sum(money) desc) as revenue_rank
from coffe_sales
group by coffee_name;

-- Find cumulative sales over time (using Date)
select
  Date,
  sum(money) as daily_sales,
  sum(sum(money)) over (order by Date) as cumulative_sales
from coffe_sales
group by Date
order by Date;

-- Calculate moving average of sales over last 3 days
select
  Date,
  sum(money) as daily_sales,
  avg(sum(money)) over (order by Date rows between 2 preceding and current row) as moving_avg_3d
from coffe_sales
group by Date
order by Date;

-- Identify peak sales day for each month
with daily_sales as (
  select Month_name, Monthsort, Date, sum(money) as total_sales,
  row_number() over (partition by Month_name order by sum(money) desc) as rn
  from coffe_sales
  group by Month_name, Monthsort, Date
)
select Month_name, Date, total_sales
from daily_sales
where rn = 1
order by Monthsort;

-- Find percentage contribution of each coffee_name to total sales
select
  coffee_name,
  sum(money) as total_sales,
  100 * sum(money) / sum(sum(money)) over () as pct_contribution
from coffe_sales
group by coffee_name;


-- Detect hours where sales are above average. 
with above_avg (
  select hour_of_day, round(avg(money),2) as avg_sales, round(sum(money),2) as total_sales
  from coffe_sales
  group by hour_of_day
)   
select * from above_avg
where avg_sales < total_sales;

-- Compare weekday vs weekend sales
select
  case when Weekday in ('Sat', 'Sun') then 'Weekend' else 'Weekday' end as day_type,
  round(sum(money),2) as total_sales,
  count(*) as transaction_count
from coffe_sales
group by day_type;

-- Find most preferred payment method (cash_type) per coffee_name
with pay as (
  select coffee_name, cash_type, count(*) as count,
    row_number() over (partition by coffee_name order by count(*) desc) as rn
  from coffe_sales
  group by coffee_name, cash_type
)
select coffee_name, cash_type as preferred_cash_type, count as transaction_count
from pay
where rn = 1;

-- 	Use window functions to rank sales within each month. 
select Month_name, coffee_name, money,
  dense_rank() over (partition by Month_name order by money desc) as sales_rank
from coffe_sales
order by Monthsort, sales_rank;